In [ ]:
## Load libraries
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import TaxSolver as tx
from TaxSolver.data_wrangling.data_loader import DataLoader
from TaxSolver.constraints.budget_constraint import BudgetConstraint
from TaxSolver.constraints.income_constraint import IncomeConstraint
from TaxSolver.constraints.marginal_pressure_constraint import MarginalPressureConstraint
from TaxSolver.objective import SequentialMixedObjective
from TaxSolver.backend.gurobi_backend import GurobiBackend

# Import the NLD rule book from the paper folder (local stand-alone copy)
from nld_rule_book import NLDRuleBook, setup_nl_optimization, get_mutually_exclusive_constraints

from copy import deepcopy

In [ ]:
def load_nl_data(data_path: str) -> dict:
    """Load NL data with the appropriate column mappings."""
    dl = DataLoader(
        path=data_path,
        income_before_tax="income_before_tax",
        income_after_tax="income_after_tax",
        weight="weight",
        id="id",
        hh_id="hh_id",
        mirror_id="mirror_id",
        input_vars=[
            "i_number_of_kids_0_5",
            "i_number_of_kids_6_11",
            "i_number_of_kids_12_15",
            "i_number_of_kids_16_17",
            "i_number_of_kids",
            "i_monthly_rent",
            "i_assets",
            "i_partner_income",
            "i_other_income",
            "i_woz",
            "i_mortgage_interest",
        ],
        group_vars=["fiscal_partner", "partner_type_of_income"],
    )
    return dl.households


def get_household_table(tax_solver) -> pd.DataFrame:
    """Extract household-level data with marginal pressures after solving."""
    if not tax_solver.solved:
        raise ValueError("Tax solver must be solved before extracting household table")
    
    household_data = []
    
    for hh_id, household in tax_solver.households.items():
        # Calculate household income before tax
        household_income_before_tax = sum(
            person["income_before_tax"] for person in household.members
        )
        
        # Get marginal pressures for all members
        marginal_pressures = []
        net_incomes = []
        
        for person in household.members:
            # Get marginal pressure value
            if person.new_marginal_rate is not None:
                mp = tax_solver.backend.get_value(person.new_marginal_rate)
                marginal_pressures.append(mp)
            else:
                marginal_pressures.append(0.0)
            
            # Get net income
            if person.new_net_income is not None:
                net_income = tax_solver.backend.get_value(person.new_net_income)
                net_incomes.append(net_income)
            else:
                # Fallback: calculate from income_before_tax + tax_balance
                income_before = person["income_before_tax"]
                if person.tax_balance is not None:
                    tax_bal = tax_solver.backend.get_value(person.tax_balance)
                    net_income = income_before + tax_bal
                else:
                    net_income = income_before
                net_incomes.append(net_income)
        
        # Calculate household-level aggregates
        new_marginal_pressure = max(marginal_pressures) if marginal_pressures else 0.0
        new_net_income = sum(net_incomes)
        
        # Get household benefits if available
        if hasattr(household, 'household_benefits') and household.household_benefits is not None:
            household_benefits = tax_solver.backend.get_value(household.household_benefits)
        else:
            household_benefits = 0.0
        
        # Add household-level data
        hh_data = {
            "hh_id": hh_id,
            "household_income_before_tax": household_income_before_tax,
            "new_marginal_pressure": new_marginal_pressure,
            "new_net_income": new_net_income,
            "new_household_benefits": household_benefits,
            "household_size": len(household.members),
        }
        
        # Add person-level data (first person's data as representative)
        if household.members:
            first_person = household.members[0]
            # Add some key person-level variables
            for key in ["weight", "k_couple", "k_single", "i_number_of_kids"]:
                if key in first_person.data:
                    hh_data[key] = first_person.data[key]
        
        household_data.append(hh_data)
    
    return pd.DataFrame(household_data)


def run_nl_optimization(
    households: dict,
    max_marginal_pressure: float = 0.65,
    max_budget_increase: float = None,  # Max additional spending (government loses revenue)
    min_budget_increase: float = None,  # Min spending change (negative = must collect more)
    income_constraint_value: float = 0.05,
    include_tags: list = None,
    exclude_tags: list = None,
    k_groups: list = None,
    time_limit: int = 30 * 60,
):
    """Run the NL tax optimization at a given marginal-pressure cap."""
    if include_tags is None:
        include_tags = ["basic"]  # Core rules
    if exclude_tags is None:
        exclude_tags = ["verzilverbaar", "protect_other_koopkracht_groups"]
    if k_groups is None:
        k_groups = NLDRuleBook.default_k_groups()
    
    print(f"Configuration:")
    print(f"  - include_tags: {include_tags}")
    print(f"  - exclude_tags: {exclude_tags}")
    print(f"  - k_groups: {k_groups}")
    print(f"  - max_marginal_pressure: {max_marginal_pressure}")
    print(f"  - max_budget_increase: {max_budget_increase}")
    print(f"  - min_budget_increase: {min_budget_increase}")
    print(f"  - income_constraint: {income_constraint_value}")
    
    backend = GurobiBackend()
    backend.model.setParam('TimeLimit', time_limit)
    backend.model.setParam('NumericFocus', 2)
    
    solver_households = deepcopy(households)
    tax_solver = tx.TaxSolver(households=solver_households, backend=backend)
    
    rules = setup_nl_optimization(
        tax_solver=tax_solver,
        k_groups=k_groups,
        include_tags=include_tags,
        exclude_tags=exclude_tags,
        add_main_brackets=True,   # 13 main brackets with weight=1
        add_k_group_brackets=True,  # 30 k-group brackets with weight=4
        add_household_brackets=False,  # Avoids double-taxing couples
    )
    print(f"\nAdded {len(rules)} total rules:")
    
    current_tax_revenue = sum(
        hh.members[0]["weight"] * sum(
            p["income_before_tax"] - p["income_after_tax"] for p in hh.members
        )
        for hh in solver_households.values()
    )
    print(f"  - current_tax_revenue (weighted): {current_tax_revenue:,.2f}")
    
    # max_budget_increase: positive = government can collect LESS (lose revenue)
    # min_budget_increase: negative = government can collect MORE (gain revenue)
    # Note: With 1% tolerance, theoretical range becomes ±2.5%
    if max_budget_increase is None:
        max_budget_increase = current_tax_revenue * 0.015  # Allow 1.5% revenue decrease
    if min_budget_increase is None:
        min_budget_increase = -current_tax_revenue * 0.015  # Allow 1.5% revenue increase
    
    print(f"  - max_budget_increase (1.5% loss): {max_budget_increase:,.2f}")
    print(f"  - min_budget_increase (1.5% gain): {min_budget_increase:,.2f}")
    
    income_constraint = IncomeConstraint(income_constraint_value, list(solver_households.values()))
    budget_constraint = BudgetConstraint(
        "all_households",
        list(solver_households.values()),
        max_bln_mut_expenditure=max_budget_increase,
        min_bln_mut_expenditure=min_budget_increase,
    )
    marginal_pressure_constraint = MarginalPressureConstraint(max_marginal_pressure)
    
    tax_solver.add_constraints([
        income_constraint,
        budget_constraint,
        marginal_pressure_constraint,
    ])
    
    # These ensure certain rules can't be active at the same time
    mutually_exclusive_constraints = get_mutually_exclusive_constraints()
    tax_solver.add_constraints(mutually_exclusive_constraints)
    print(f"Added {len(mutually_exclusive_constraints)} mutually exclusive constraints")
    
    # Sequential objective: minimize revenue loss first, then rule count
    budget_tolerance = current_tax_revenue * 0.01  # 1% tolerance
    objective = SequentialMixedObjective(
        budget_constraint,
        objectives={"budget": 2, "complexity": 1},
        tolerances={"budget": budget_tolerance, "complexity": 10},
    )
    tax_solver.add_objective(objective)
    print(f"  - budget_tolerance (1%): {budget_tolerance:,.2f}")
    
    print(f"\nSolving with max_marginal_pressure={max_marginal_pressure}...")
    tax_solver.solve()
    
    return tax_solver


In [ ]:
# Load the NL data
data_path = "./data/NL_persons_headers_preprocessed_equal_weights.xlsx"
households = load_nl_data(data_path)
print(f"Total number of households loaded: {len(households)}")


In [ ]:
# Batch optimization: multiple marginal pressure levels
marginal_pressure_levels = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
results = {}
summary_data = []
income_constraint_value = 0.05
# First, calculate current system stats (baseline without reform)
print("=" * 60)
print("Calculating current system baseline...")
print("=" * 60)

# Calculate baseline: current tax balance from households
current_expenditures = sum(
    hh.members[0]["weight"] * sum(
        p["income_after_tax"] - p["income_before_tax"] for p in hh.members
    )
    for hh in households.values()
)

# For current system, count the number of sq_a_ columns that have non-zero values
first_person = next(iter(households.values())).members[0]
sq_rules = [k for k in first_person.data.keys() if k.startswith("sq_a_")]
current_active_rules = len(sq_rules)  # Approximate count of current system rules

summary_data.append({
    "Cap": "Current",
    "Marginal Pressure Cap": None,
    "Active Rules": current_active_rules,
    "Complexity (Rule Weight)": None,  # Not applicable for current system
    "Budget Spend": 0,  # Baseline is 0 by definition
    "Tax Revenue Change (%)": 0,
    "Status": "Baseline"
})

print(f"Current system baseline expenditures: {current_expenditures:,.2f}")
print(f"Current system approximate rule count: {current_active_rules}")

# Run optimizations for each marginal pressure level
for max_pressure in marginal_pressure_levels:
    print(f"\n{'='*60}")
    print(f"Running optimization for max_marginal_pressure = {max_pressure:.0%}")
    print(f"{'='*60}")
    
    try:
        tax_solver = run_nl_optimization(
            households=households,
            max_marginal_pressure=max_pressure,
            # Uses default budget constraints: +10% max loss, -5% max gain
            income_constraint_value=income_constraint_value,
        )
        
        r_and_r = tax_solver.rules_and_rates_table()
        
        # Calculate summary stats
        active_rules = int(r_and_r['b'].sum())
        complexity = int((r_and_r['b'] * r_and_r['weight']).sum())
        
        # Get budget spend from the budget constraint
        budget_spend = None
        for constraint in tax_solver.constraints:
            if hasattr(constraint, 'spend'):
                budget_spend = tax_solver.backend.get_value(constraint.spend)
                break
        
        # Calculate tax revenue change percentage
        tax_revenue_change_pct = (budget_spend / abs(current_expenditures) * 100) if current_expenditures != 0 else 0
        
        results[max_pressure] = {
            'solver': tax_solver,
            'rates': r_and_r,
            'active_rules': active_rules,
            'complexity': complexity,
            'budget_spend': budget_spend,
        }
        
        summary_data.append({
            "Cap": f"{int(max_pressure * 100)}%",
            "Marginal Pressure Cap": max_pressure,
            "Active Rules": active_rules,
            "Complexity (Rule Weight)": complexity,
            "Budget Spend": budget_spend,
            "Tax Revenue Change (%)": tax_revenue_change_pct,
            "Status": "Success"
        })
        
        # Save rules and rates to Excel
        output_file = f"./systems/case_nl_reform_{int(max_pressure * 100)}_{int(income_constraint_value * 100)}.xlsx"
        r_and_r.to_excel(output_file, index=False)
        print(f"Saved rules and rates to {output_file}")
        
        # Save household table with marginal pressures
        household_table = get_household_table(tax_solver)
        household_table = household_table.loc[household_table["new_marginal_pressure"] < 1]
        household_output_file = f"./systems/case_nl_reform_{int(max_pressure * 100)}_households.xlsx"
        household_table.to_excel(household_output_file, index=False)
        print(f"Saved household data to {household_output_file}")
        
        print(f"  Active rules: {active_rules}")
        print(f"  Complexity (rule weight): {complexity}")
        print(f"  Budget spend: {budget_spend:,.2f}")
        print(f"  Tax revenue change: {tax_revenue_change_pct:.2f}%")
        
    except Exception as e:
        print(f"Failed for max_pressure={max_pressure}: {e}")
        results[max_pressure] = None
        summary_data.append({
            "Cap": f"{int(max_pressure * 100)}%",
            "Marginal Pressure Cap": max_pressure,
            "Active Rules": None,
            "Complexity (Rule Weight)": None,
            "Budget Spend": None,
            "Tax Revenue Change (%)": None,
            "Status": f"Failed: {str(e)[:50]}"
        })

# Create and save summary table
summary_df = pd.DataFrame(summary_data)
summary_df.to_excel(f"./systems/case_nl_system_comparison_{int(income_constraint_value * 100)}.xlsx", index=False)

print(f"\n{'='*60}")
print("Batch optimization complete!")
print(f"{'='*60}")
print(f"\nSummary table saved to ./systems/case_nl_system_comparison_{int(income_constraint_value * 100)}.xlsx")
print("\nSummary:")
print(summary_df.to_string(index=False))

In [ ]:
# =====================================================================
# MANUSCRIPT FIGURES
#   (a) current system + (b) reform comparison -> case_nl_combined_figure.png
#   65% vs 75% reform hexbins               -> case_nl_65_75_comparison.png
# Sequential quantities (hexbin density, complexity) use viridis; budget
# bands are neutral greys (grayscale- and colorblind-safe).
# =====================================================================
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np

income_constraint_value = 0.05
SEQ_CMAP = 'viridis'

# Load the raw data to get current system marginal pressures
raw_data = pd.read_excel("./data/NL_persons_headers_preprocessed_equal_weights.xlsx")

current_mp_by_hh = raw_data.groupby('hh_id').agg({
    'household_income_before_tax': 'first',
    'sq_m_total': 'max',
    'weight': 'first'
}).reset_index()
current_mp_by_hh = current_mp_by_hh[current_mp_by_hh['sq_m_total'] < 1]
current_mp_by_hh = current_mp_by_hh[current_mp_by_hh['sq_m_total'] > -0.5]

household_table_65 = pd.read_excel("./systems/case_nl_reform_65_households.xlsx")
household_table_75 = pd.read_excel("./systems/case_nl_reform_75_households.xlsx")
household_table_65 = household_table_65.loc[household_table_65["new_marginal_pressure"] < 1]
household_table_75 = household_table_75.loc[household_table_75["new_marginal_pressure"] < 1]

extent = (0, 250_000, 0, 0.9)
gridsize = 45
mincnt = 1
tick_values = [1, 2, 5, 10, 20, 50, 100, 200, 500]

def euro_format(x, pos):
    return f"\u20ac{x/1000:,.0f}K"

def density_hexbin(fig, ax, x, y):
    """Log-density hexbin with a viridis colormap and formatted colorbar."""
    hb = ax.hexbin(x, y, gridsize=gridsize, cmap=SEQ_CMAP, mincnt=mincnt,
                   extent=extent, bins='log', edgecolors='black', linewidths=0.2)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.xaxis.set_major_formatter(FuncFormatter(euro_format))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.tick_params(axis="both", labelsize=10)
    ax.set_xlabel("Household income before tax (\u20ac)", fontsize=11)
    ax.set_ylabel("Marginal pressure", fontsize=11)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    cbar = fig.colorbar(hb, ax=ax, shrink=0.85, pad=0.02)
    cbar.set_label("Sampled households", fontsize=9)
    arr = hb.get_array()
    if len(arr) > 0:
        valid = [v for v in tick_values if 1 <= v <= arr.max()]
        if valid:
            cbar.set_ticks(valid)
            cbar.set_ticklabels([str(int(v)) for v in valid])
    return hb

# ============================================================
# Figure 1: (a) current system hexbin + (b) reform comparison
# ============================================================
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

density_hexbin(fig1, ax1, current_mp_by_hh['household_income_before_tax'],
               current_mp_by_hh['sq_m_total'])

summary_df = pd.read_excel(f"./systems/case_nl_system_comparison_{int(income_constraint_value * 100)}.xlsx")
reform_data = summary_df[summary_df['Status'] == 'Success'].copy()

current_complexity = 101
complexity_min = reform_data['Complexity (Rule Weight)'].min()
complexity_max = 105

sc = ax2.scatter(
    reform_data['Marginal Pressure Cap'] * 100,
    -reform_data['Tax Revenue Change (%)'],
    c=reform_data['Complexity (Rule Weight)'],
    cmap=SEQ_CMAP, vmin=complexity_min, vmax=complexity_max,
    s=150, edgecolors='black', linewidths=1, zorder=5,
)
ax2.scatter([85], [0], c=[current_complexity], cmap=SEQ_CMAP,
            vmin=complexity_min, vmax=complexity_max,
            s=150, edgecolors='black', linewidths=1.5, marker='o', zorder=6)
ax2.annotate('Current\nsystem', xy=(84.3, -0.15), xytext=(80, -1),
             fontsize=9, ha='center', va='top', color='black',
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='black', lw=1.2),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1.2))

ax2.set_xlim(50, 88)
ax2.set_ylim(-4, 4)
ax2.set_xlabel("Marginal pressure cap", fontsize=11)
ax2.set_ylabel("Tax revenue change (%)", fontsize=11)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)

# Budget bands as neutral greys: dark = +/-1.5% hard band,
# light = further +/-1% second-stage tolerance
ax2.axhspan(-1.5, 1.5, alpha=0.22, color='0.55', zorder=1)
ax2.axhspan(-2.5, 2.5, alpha=0.12, color='0.55', zorder=0)

ax2.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax2.tick_params(axis="both", labelsize=10)
ax2.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

cbar2 = fig1.colorbar(sc, ax=ax2, shrink=0.85, pad=0.02)
cbar2.set_label("Weighted rule count", fontsize=9)

fig1.tight_layout()
fig1.savefig("./output/figs/case_nl_combined_figure.png", dpi=300, bbox_inches='tight')
print("Saved ./output/figs/case_nl_combined_figure.png")
plt.show()

# ============================================================
# Figure 2: 65% vs 75% cap reform hexbins
# ============================================================
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 6))

for ax, table, cap in [(ax3, household_table_65, 0.65), (ax4, household_table_75, 0.75)]:
    density_hexbin(fig2, ax, table['household_income_before_tax'],
                   table['new_marginal_pressure'])
    ax.axhline(cap, color="black", ls="--", lw=1.5, alpha=0.9)
    ax.annotate(f'{cap:.0%} cap', xy=(200_000, cap), xytext=(200_000, cap + 0.08),
                fontsize=10, ha='center', va='bottom', color='black',
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='black', lw=1.2),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1.2))

fig2.tight_layout()
fig2.savefig("./output/figs/case_nl_65_75_comparison.png", dpi=300, bbox_inches='tight')
print("Saved ./output/figs/case_nl_65_75_comparison.png")
plt.show()

In [ ]:
# ============================================================
# Figure 11: distributional incidence (fig: case_nl_incidence)
#   change in net household income by decile of household income
#   before tax, for both headline reforms
# ============================================================
from matplotlib.ticker import MultipleLocator

def load_pct_change(cap):
    """Per-household %% change in net income vs. the status quo, matching
    the solver's own income-guarantee basis exactly: old_household_income =
    sum(income_after_tax) (see IncomeConstraint/Household.old_household_income
    in the TaxSolver source), new = new_net_income + new_household_benefits.
    Households with a near-zero status-quo baseline are excluded (percentage
    change is not meaningful there)."""
    hh_sq = raw_data.drop_duplicates("hh_id")[
        ["hh_id", "household_income_before_tax", "household_income_after_tax"]
    ]
    hh = pd.read_excel(f"./systems/case_nl_reform_{cap}_households.xlsx")
    m = hh.merge(hh_sq, on="hh_id", suffixes=("", "_sq"))
    m["new_net_household_income"] = m["new_net_income"] + m["new_household_benefits"]
    m = m[m["household_income_after_tax"] > 500].copy()
    m["pct_change"] = (
        (m["new_net_household_income"] - m["household_income_after_tax"])
        / m["household_income_after_tax"] * 100
    )
    m["decile"] = pd.qcut(m["household_income_before_tax"].rank(method="first"), 10, labels=False) + 1
    return m


def box_stats(vals):
    """Standard Tukey (1.5*IQR) box-and-whisker stats, no fliers."""
    q1, med, q3 = np.percentile(vals, [25, 50, 75])
    iqr = q3 - q1
    lo_fence, hi_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    whislo = vals[vals >= lo_fence].min() if (vals >= lo_fence).any() else vals.min()
    whishi = vals[vals <= hi_fence].max() if (vals <= hi_fence).any() else vals.max()
    return dict(q1=q1, med=med, q3=q3, whislo=whislo, whishi=whishi,
                p90=np.percentile(vals, 90), fliers=[])


INCIDENCE_COLOR = "#8FD19E"
INCIDENCE_YLIM = (-10, 40)

fig3, axes3 = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, cap, title in zip(axes3, ["65", "75"], ["Reform (65% cap)", "Reform (75% cap)"]):
    df = load_pct_change(cap)
    stats = [box_stats(df.loc[df.decile == d, "pct_change"].values) for d in range(1, 11)]

    # Clip whishi at the box top for out-of-range boxes so bxp draws no
    # upper whisker; an arrow + p90 annotation communicates the true extent.
    bxp_stats = []
    for d, s in zip(range(1, 11), stats):
        s2 = dict(s)
        if s["whishi"] > INCIDENCE_YLIM[1]:
            s2["whishi"] = s["q3"]
        bxp_stats.append({**s2, "label": str(d)})

    bp = ax.bxp(bxp_stats, positions=range(1, 11), widths=0.6, patch_artist=True,
                showfliers=False)
    for box in bp["boxes"]:
        box.set_facecolor(INCIDENCE_COLOR)
        box.set_edgecolor("black")
    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(2)
    for element in ("whiskers", "caps"):
        for line in bp[element]:
            line.set_color("black")

    for d, s in zip(range(1, 11), stats):
        if s["whishi"] > INCIDENCE_YLIM[1]:
            ax.annotate(
                "", xy=(d, INCIDENCE_YLIM[1] - 0.3), xytext=(d, s["q3"]),
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.5),
            )
            ax.text(d + 0.15, INCIDENCE_YLIM[1] - 4, f"p90: {s['p90']:+.0f}%", fontsize=10,
                     bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"))

    ax.axhline(0, color="gray", linewidth=1)
    ax.axhline(-5, color="black", linestyle="--", linewidth=1.5)
    ax.text(9.6, -5.9, "5% income floor", ha="right", va="top", fontsize=10)
    ax.set_ylim(*INCIDENCE_YLIM)
    ax.set_xlabel("Decile of household income before tax", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.yaxis.set_major_locator(MultipleLocator(10))
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes3[0].set_ylabel("Change in net household income (%)", fontsize=12)
fig3.tight_layout()
fig3.savefig("./output/figs/case_nl_incidence.png", dpi=300, bbox_inches='tight')
print("Saved ./output/figs/case_nl_incidence.png")
plt.show()
